# RQ2 - Collecting the four forecasters (Seasonal Naive, Persistence, LSTM, Hybrid)

This notebook reads the prediction files written by the four RQ2 notebooks (added as inputs), checks that all of them use the same benchmark, stacks them into one table (`rq2_combined_predictions.csv`, used by the RQ3 notebook) and reports the Test results of every model under both training runs.

Run 1: fit on Train, tune and early-stop on Val, evaluate on Test. Run 2: refit on Train+Val, evaluate on the same Test block (2020-01-16 to the end of the data).

In [ ]:
MODEL_TAG, MODEL_NAME = 'rq2_aggregate', 'RQ2 aggregate'

In [ ]:
# ====================== SHARED CONFIG (identical in every RQ2/RQ3 notebook) ======================
import os, sys, glob, json, random, warnings, subprocess, time, hashlib, platform, base64, gzip, io
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from numpy.lib.stride_tricks import sliding_window_view
from pandas.tseries.holiday import USFederalHolidayCalendar
from sklearn.metrics import mean_absolute_error, mean_squared_error
pd.set_option('display.width', 250, 'display.max_columns', None, 'display.max_rows', 500)

FAST_MODE   = False        # True = smoke test only
SEED        = 42
SITES       = [1, 2]       # one hourly series (and one model) per site; site 19 is dropped (too sparse)
HORIZONS    = [1, 24]      # H=1 next hour, H=24 one day ahead (direct strategy)

# ---- Split requested by the author: Train -> 31/10/2019, Val -> 15/01/2020, Test = 16/01/2020 -> end of data ----
TRAIN_END   = '2019-10-31 23:00'
VAL_END     = '2020-01-15 23:00'
# Two training protocols:
#   run1 : fit on Train, early-stop / select on Val, evaluate on Test        (train - val - test)
#   run2 : refit on Train+Val with the settings chosen in run1, evaluate on Test   (train(train+val) - test)

MAX_LOOKBACK = 168         # longest history window (hours) any model may read
WARMUP_H     = 24 * 28     # first 4 weeks of each series are never targets (672 h of history needed by seasonal_mean_4w)
GAP_DAYS     = 7           # >= 7 consecutive hours-days with zero energy = data outage, not demand
POST_GAP_H   = 24 * 28     # after every outage the next 4 weeks are not targets (lags would read the artificial zeros)
MAPE_MIN     = 0.5         # kWh; MAPE/sMAPE only on hours with at least this load
N_BOOT       = 1000
BLOCK_H      = 24 * 7
if FAST_MODE: N_BOOT = 100
random.seed(SEED); np.random.seed(SEED)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': .3,
                     'axes.spines.top': False, 'axes.spines.right': False})

# ---- The 16 common features (CFV) used by EVERY forecasting model ----
FEATS_COMMON = ['lag_H', 'lag_D', 'lag_168', 'roll_mean_24', 'roll_max_24', 'seasonal_mean_4w',
                'n_plugged_lagH', 'n_charging_lagH', 'n_arrivals_lagH', 'charging_share_lagH',
                'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'is_weekend', 'is_holiday']
assert len(FEATS_COMMON) == 16
FEATS_HASH = hashlib.md5('|'.join(FEATS_COMMON).encode()).hexdigest()[:10]
CAL = ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'is_weekend', 'is_holiday']
HIST_COLS = ['y_s', 'plug_s', 'chg_s', 'arr_s']          # 4 dynamic channels for recurrent nets

In [ ]:
# ---------- Output folder + data file ----------
OUT_DIR = '/kaggle/working' if os.path.exists('/kaggle/working') else './outputs_' + MODEL_TAG
FIG_DIR = os.path.join(OUT_DIR, 'figures'); os.makedirs(FIG_DIR, exist_ok=True)
def savefig(name):
    plt.savefig(os.path.join(FIG_DIR, f'{MODEL_TAG}_{name}.png'), dpi=140, bbox_inches='tight'); plt.show()
def resolve_data():
    pats = ['/kaggle/input/**/acn_data*.xlsx', '/kaggle/input/**/acn*.xlsx', './acn*.xlsx', '/mnt/user-data/uploads/**/acn*.xlsx']
    for p in pats:
        hits = sorted(glob.glob(p, recursive=True))
        if hits: return hits[0]
    raise FileNotFoundError('acn*.xlsx not found. Add the ACN dataset as notebook input.')
DATA_PATH = resolve_data()
print('Model:', MODEL_NAME, '| data:', DATA_PATH, '| python', sys.version.split()[0])

# ---------- Read + clean sessions ----------
try: raw = pd.read_excel(DATA_PATH, sheet_name='full_data')
except ValueError: raw = pd.read_excel(DATA_PATH)
for c in ['connectionTime', 'disconnectTime', 'doneChargingTime']: raw[c] = pd.to_datetime(raw[c])
sess = raw.drop_duplicates(['stationID', 'connectionTime']).copy()
sess['done'] = sess['doneChargingTime'].clip(upper=sess['disconnectTime'])
sess = sess[(sess.kWhDelivered > 0) & (sess.disconnectTime > sess.connectionTime)]
sess = sess[sess.siteID.isin(SITES)].reset_index(drop=True)
print(f'sessions after cleaning (sites {SITES}): {len(sess):,} / raw {len(raw):,}')

# ---------- Hourly load series per site ----------
def spread_energy(s):
    c = s.connectionTime.values.astype('datetime64[s]'); e = s.done.values.astype('datetime64[s]')
    e = np.maximum(e, c + np.timedelta64(60, 's'))
    c0 = c.astype('datetime64[h]'); e0 = (e - np.timedelta64(1, 's')).astype('datetime64[h]')
    n = ((e0 - c0) // np.timedelta64(1, 'h')).astype(int) + 1
    idx = np.repeat(np.arange(len(s)), n)
    k = np.arange(n.sum()) - np.repeat(np.cumsum(n) - n, n)
    hs = c0[idx] + k.astype('timedelta64[h]'); he = hs + np.timedelta64(1, 'h')
    ov = (np.minimum(e[idx], he) - np.maximum(c[idx], hs)) / np.timedelta64(1, 's')
    tot = ((e - c) / np.timedelta64(1, 's'))[idx]
    return pd.Series(s.kWhDelivered.values[idx] * ov / tot, index=pd.DatetimeIndex(hs)).groupby(level=0).sum()

def hourly_site(s):
    y = spread_energy(s)
    grid = pd.date_range(s.connectionTime.min().floor('h'), s.done.max().ceil('h'), freq='h', name='time')
    out = pd.DataFrame(index=grid); out['y'] = y.reindex(grid).fillna(0.0)
    ends = (grid + pd.Timedelta(hours=1)).values
    conn = np.sort(s.connectionTime.values); disc = np.sort(s.disconnectTime.values); done = np.sort(s.done.values)
    out['n_plugged_end']  = np.searchsorted(conn, ends, 'right') - np.searchsorted(disc, ends, 'right')
    out['n_charging_end'] = np.searchsorted(conn, ends, 'right') - np.searchsorted(done, ends, 'right')
    out['n_arrivals'] = pd.Series(1, index=s.connectionTime.dt.floor('h')).groupby(level=0).sum().reindex(grid).fillna(0).values
    return out

hol = USFederalHolidayCalendar().holidays('2018-01-01', '2021-12-31')
def add_calendar(Hs):
    ix = Hs.index
    Hs['hour_sin'] = np.sin(2*np.pi*ix.hour/24);     Hs['hour_cos'] = np.cos(2*np.pi*ix.hour/24)
    Hs['dow_sin']  = np.sin(2*np.pi*ix.dayofweek/7); Hs['dow_cos']  = np.cos(2*np.pi*ix.dayofweek/7)
    Hs['is_weekend'] = (ix.dayofweek >= 5).astype(float)
    Hs['is_holiday'] = ix.normalize().isin(hol).astype(float)     # US federal calendar, NOT the file's is_holiday column
    return Hs

T_END, V_END = pd.Timestamp(TRAIN_END), pd.Timestamp(VAL_END)
HOURLY, GAP_RUNS, SCALE, THR90, MASE_SCALE = {}, {}, {}, {}, {}
for sid in SITES:
    s = sess[sess.siteID == sid]
    Hs = hourly_site(s)
    assert abs(Hs.y.sum() - s.kWhDelivered.sum()) < 1e-3, 'kWh not conserved'
    inactive = (Hs.y == 0); grp = (inactive != inactive.shift()).cumsum()
    Hs['in_gap'] = inactive & (inactive.groupby(grp).transform('size') >= GAP_DAYS * 24)
    _g = (Hs.in_gap != Hs.in_gap.shift()).cumsum()
    GAP_RUNS[sid] = [(g.index.min(), g.index.max()) for _, g in Hs[Hs.in_gap].groupby(_g[Hs.in_gap])]
    post = np.zeros(len(Hs), bool)
    for a, b in GAP_RUNS[sid]:
        j = Hs.index.get_loc(b); post[j + 1: j + 1 + POST_GAP_H] = True
    Hs['post_gap'] = post
    Hs['valid'] = (~Hs.in_gap.values) & (~post) & (np.arange(len(Hs)) >= WARMUP_H)   # identical target hours for every model
    Hs = add_calendar(Hs)
    tr_ok = (Hs.index <= T_END) & Hs.valid.values
    SCALE[sid] = {c: max(float(np.quantile(Hs.loc[tr_ok, c], .99)), 1e-6) for c in ['y', 'n_plugged_end', 'n_charging_end', 'n_arrivals']}
    Hs['y_s'] = Hs.y / SCALE[sid]['y']; Hs['plug_s'] = Hs.n_plugged_end / SCALE[sid]['n_plugged_end']
    Hs['chg_s'] = Hs.n_charging_end / SCALE[sid]['n_charging_end']; Hs['arr_s'] = Hs.n_arrivals / SCALE[sid]['n_arrivals']
    THR90[sid] = float(np.quantile(Hs.loc[tr_ok, 'y'], .90))
    d24 = (Hs.y - Hs.y.shift(24)).abs()[tr_ok & Hs.valid.shift(24, fill_value=False).values].dropna()
    MASE_SCALE[sid] = float(d24.mean())
    HOURLY[sid] = Hs
    print(f'site {sid}: {len(Hs):,} h | {Hs.index.min()} -> {Hs.index.max()} | valid target hours {int(Hs.valid.sum()):,} | P90(train)={THR90[sid]:.2f} kWh')
    for a, b in GAP_RUNS[sid]: print(f'    outage {a} -> {b} ({(b-a).days+1} days)')

# ---------- Split (time based) ----------
def split_of(t):
    t = np.asarray(t)
    return np.select([t <= T_END, t <= V_END], ['train', 'val'], default='test')
def target_index(site): return np.flatnonzero(HOURLY[site].valid.values)

_rows = []
for sid in SITES:
    ti = target_index(sid); spl = split_of(HOURLY[sid].index.values[ti])
    for nm in ['train', 'val', 'test']:
        m = spl == nm
        _rows.append(dict(site=sid, split=nm, n_target_hours=int(m.sum()), first=HOURLY[sid].index[ti][m].min(), last=HOURLY[sid].index[ti][m].max()))
SPLIT_TBL = pd.DataFrame(_rows).set_index(['site', 'split']); print(SPLIT_TBL.to_string())
BENCH_FINGERPRINT = hashlib.md5('|'.join(map(str, [SITES, HORIZONS, TRAIN_END, VAL_END, MAX_LOOKBACK, WARMUP_H, GAP_DAYS,
                                  POST_GAP_H, MAPE_MIN, FEATS_HASH, SEED])).encode()).hexdigest()[:12]
print('BENCHMARK FINGERPRINT =', BENCH_FINGERPRINT, '| features hash =', FEATS_HASH)

# ---------- 16 common features ----------
def build_common_features(Hs, h):
    """16 columns; every value uses information up to hour t-h only, or is a deterministic function of the calendar."""
    y = Hs.y; D = 24 if h < 24 else 48; f = pd.DataFrame(index=Hs.index)
    f['lag_H'] = y.shift(h); f['lag_D'] = y.shift(D); f['lag_168'] = y.shift(168)
    ys = y.shift(h)
    f['roll_mean_24'] = ys.rolling(24).mean(); f['roll_max_24'] = ys.rolling(24).max()
    f['seasonal_mean_4w'] = pd.concat([y.shift(168 * k) for k in range(1, 5)], axis=1).mean(axis=1)
    f['n_plugged_lagH'] = Hs.n_plugged_end.shift(h); f['n_charging_lagH'] = Hs.n_charging_end.shift(h)
    f['n_arrivals_lagH'] = Hs.n_arrivals.shift(h)
    f['charging_share_lagH'] = (f.n_charging_lagH / f.n_plugged_lagH.replace(0, np.nan)).fillna(0.0)
    for c in CAL: f[c] = Hs[c]
    return f[FEATS_COMMON]

def scale_features(X, site):
    """Same definition, scaled for neural nets (load features / P99(y); count features / P99(count); calendar unchanged)."""
    Z = X.copy(); sy = SCALE[site]['y']
    for c in ['lag_H', 'lag_D', 'lag_168', 'roll_mean_24', 'roll_max_24', 'seasonal_mean_4w']: Z[c] = Z[c] / sy
    Z['n_plugged_lagH'] /= SCALE[site]['n_plugged_end']; Z['n_charging_lagH'] /= SCALE[site]['n_charging_end']
    Z['n_arrivals_lagH'] /= SCALE[site]['n_arrivals']
    return Z.values.astype('float32')

def make_data(site, h, sequences=False):
    Hs = HOURLY[site]; tgt = target_index(site); spl = split_of(Hs.index[tgt]); L = MAX_LOOKBACK
    f = build_common_features(Hs, h)
    if sequences:
        S = Hs[HIST_COLS].values.astype('float32'); Wv = sliding_window_view(S, L, axis=0).swapaxes(1, 2)
    def pack(sel):
        X = f.iloc[sel]
        assert not X.isna().any().any(), 'NaN in features'
        d = dict(idx=sel, X=X, Xs=scale_features(X, site), y=Hs.y.values[sel], y_s=Hs.y_s.values[sel].astype('float32'))
        if sequences: d['hist'] = Wv[sel - h - L + 1].copy()
        return d
    D = {n: pack(tgt[spl == n]) for n in ['train', 'val', 'test']}
    D['dev'] = pack(tgt[np.isin(spl, ['train', 'val'])])
    return D

# ---------- Evaluation library ----------
def pred_frame(site, h, run, split, idx, pred):
    Hs = HOURLY[site]
    return pd.DataFrame({'time': Hs.index[idx], 'site': site, 'horizon': h, 'run': run, 'split': split,
                         'actual': Hs.y.values[idx], 'pred': np.clip(np.asarray(pred, float), 0, None)})

def hourly_stats(d, thr):
    a, p = d.actual.values, d.pred.values; e = p - a; m = a >= MAPE_MIN
    ape = np.where(m, np.abs(e) / np.where(m, a, 1.0), 0.0); sape = np.where(m, 2 * np.abs(e) / np.maximum(a + p, 1e-9), 0.0)
    return np.column_stack([np.ones(len(a)), a, a**2, p, p**2, a*p, np.abs(e), e, e**2, m.astype(float), ape, sape,
                            ((p >= thr) & (a >= thr)).astype(float), (p >= thr).astype(float), (a >= thr).astype(float)])
def make_hourly_fn(scale):
    def fn(s):
        n, sa, saa, sp, spp, sap, sae, se, see, nm, sape, ssape, tp, npp, nap = [s[:, i] for i in range(15)]
        with np.errstate(divide='ignore', invalid='ignore'):
            sst = saa - sa**2 / n; cv = sap - sa * sp / n; vp = spp - sp**2 / n
            return {'MAE': sae / n, 'RMSE': np.sqrt(see / n), 'R2': 1 - see / sst, 'MAPE_%': 100 * sape / nm,
                    'sMAPE_%': 100 * ssape / nm, 'WAPE_%': 100 * sae / sa, 'MASE': sae / n / scale, 'Bias_kWh': se / n,
                    'Pearson_r': cv / np.sqrt(sst * vp), 'P90_precision': tp / npp, 'P90_recall': tp / nap, 'P90_F1': 2 * tp / (npp + nap)}
    return fn
def daily_peaks(d):
    g = d.groupby(d.index.normalize()); ia, ip = g.actual.idxmax(), g.pred.idxmax()
    pk = pd.DataFrame({'act_peak': g.actual.max(), 'pred_peak': g.pred.max(), 'act_h': ia.dt.hour, 'pred_h': ip.dt.hour})
    return pk[pk.act_peak > 0]
def peak_stats(pk):
    a = pk.act_peak.values; e = (pk.pred_peak - pk.act_peak).values
    dh = (pk.pred_h - pk.act_h).abs().values; dh = np.minimum(dh, 24 - dh)
    return np.column_stack([np.ones(len(a)), a, a**2, np.abs(e), e, e**2, (dh <= 1).astype(float)])
def peak_fn(s):
    n, sa, saa, sae, se, see, hit = [s[:, i] for i in range(7)]
    with np.errstate(divide='ignore', invalid='ignore'):
        return {'Peak_MAE': sae / n, 'Peak_WAPE_%': 100 * sae / sa, 'Peak_Bias_%': 100 * se / sa, 'PeakHour_hit_pm1h_%': 100 * hit / n}
def _boot(S, fn, block, rng):
    n = len(S)
    if n / block < 8: block = max(1, block // 7)
    B = np.add.reduceat(S, np.arange(0, n, block), axis=0)
    idx = rng.integers(0, len(B), size=(N_BOOT, len(B)))
    boot = fn(B[idx].sum(axis=1)); point = fn(S.sum(axis=0, keepdims=True)); out = {}
    for k in point:
        b = boot[k][np.isfinite(boot[k])]; ok = len(b) > 2
        out[k] = (float(point[k][0]), float(b.std(ddof=1)) if ok else np.nan)
    return out
def evaluate_all(PRED, seed=SEED):
    rng = np.random.default_rng(seed); rows = []
    for (run, site, h), g in PRED.groupby(['run', 'site', 'horizon']):
        g = g.sort_values('time').set_index('time'); hfn = make_hourly_fn(MASE_SCALE[site])
        for name, d in {'Train': g[g.split == 'train'], 'Val': g[g.split == 'val'], 'Test': g[g.split == 'test']}.items():
            if len(d) < 48: continue
            res = _boot(hourly_stats(d, THR90[site]), hfn, BLOCK_H, rng); pk = daily_peaks(d)
            resp = _boot(peak_stats(pk), peak_fn, 7, rng) if len(pk) >= 8 else {}
            for src in (res, resp):
                for k, v in src.items():
                    rows.append(dict(run=run, site=site, horizon=h, group=name, metric=k, value=v[0], SE=v[1], n_hours=len(d)))
    return pd.DataFrame(rows)
MAIN = ['MAE', 'RMSE', 'R2', 'WAPE_%', 'MASE', 'Bias_kWh', 'P90_F1', 'Peak_MAE']

def standard_report(PRED, extra_json=None):
    """Save predictions + metrics + standard figures; print compact Test tables for both runs."""
    PRED = PRED.sort_values(['run', 'site', 'horizon', 'time']).reset_index(drop=True)
    PRED.to_csv(os.path.join(OUT_DIR, f'{MODEL_TAG}_predictions.csv'), index=False)
    RES = evaluate_all(PRED); RES.insert(0, 'model', MODEL_NAME)
    RES.to_csv(os.path.join(OUT_DIR, f'{MODEL_TAG}_results_long.csv'), index=False)
    T = RES[RES.group.isin(['Val', 'Test']) & RES.metric.isin(MAIN)]
    W = T.pivot_table(index=['run', 'site', 'horizon', 'group'], columns='metric', values='value')[[m for m in MAIN if m in T.metric.unique()]]
    W.to_csv(os.path.join(OUT_DIR, f'{MODEL_TAG}_results_wide.csv'))
    print(f'===== {MODEL_NAME}: Val / Test metrics =====\n', W.round(4).to_string())
    # ---- figures ----
    runs = [r for r in ['run2', 'run1'] if r in PRED.run.unique()]
    r0 = runs[0]; fig, axs = plt.subplots(len(SITES), len(HORIZONS), figsize=(7 * len(HORIZONS), 3.4 * len(SITES)), squeeze=False)
    for i, site in enumerate(SITES):
        for j, h in enumerate(HORIZONS):
            g = PRED[(PRED.run == r0) & (PRED.site == site) & (PRED.horizon == h) & (PRED.split == 'test')].set_index('time')
            g = g[g.index >= g.index.max() - pd.Timedelta(days=14)]
            axs[i, j].plot(g.index, g.actual, color='#444', lw=1.1, label='Actual'); axs[i, j].plot(g.index, g.pred, color='#D55E00', lw=1.1, label='Forecast')
            axs[i, j].set(title=f'{MODEL_NAME} | site {site} | H={h}: last 14 test days', ylabel='kWh per hour'); axs[i, j].legend(fontsize=7)
            axs[i, j].tick_params(axis='x', rotation=20, labelsize=7)
    plt.tight_layout(); savefig('F1_forecast_window')
    fig, axs = plt.subplots(len(SITES), len(HORIZONS), figsize=(4.2 * len(HORIZONS), 3.8 * len(SITES)), squeeze=False)
    for i, site in enumerate(SITES):
        for j, h in enumerate(HORIZONS):
            g = PRED[(PRED.run == r0) & (PRED.site == site) & (PRED.horizon == h) & (PRED.split == 'test')]
            axs[i, j].scatter(g.actual, g.pred, s=4, alpha=.25, color='#0072B2'); m = max(g.actual.max(), g.pred.max())
            axs[i, j].plot([0, m], [0, m], 'k--', lw=.8); axs[i, j].set(title=f'site {site} | H={h}', xlabel='Actual kWh', ylabel='Forecast kWh')
    plt.suptitle(f'{MODEL_NAME}: actual vs forecast (Test)', y=1.02); plt.tight_layout(); savefig('F2_scatter')
    fig, axs = plt.subplots(1, len(SITES), figsize=(5.5 * len(SITES), 3.4), squeeze=False)
    for i, site in enumerate(SITES):
        for h, c in zip(HORIZONS, ['#009E73', '#CC79A7']):
            g = PRED[(PRED.run == r0) & (PRED.site == site) & (PRED.horizon == h) & (PRED.split == 'test')]
            e = (g.actual - g.pred).abs().groupby(g.time.dt.hour).mean(); axs[0, i].plot(e.index, e.values, 'o-', ms=3, color=c, label=f'H={h}')
        axs[0, i].set(title=f'site {site}: MAE by hour of day (Test)', xlabel='Hour', ylabel='MAE (kWh)'); axs[0, i].legend(fontsize=7)
    plt.tight_layout(); savefig('F3_error_by_hour')
    if extra_json is not None:
        S = {'model': MODEL_NAME, 'tag': MODEL_TAG, 'fingerprint': BENCH_FINGERPRINT, 'features': FEATS_COMMON,
             'split': {'TRAIN_END': TRAIN_END, 'VAL_END': VAL_END, 'TEST': '2020-01-16 -> end of data'}, **extra_json}
        with open(os.path.join(OUT_DIR, f'{MODEL_TAG}_best_hyperparameters.json'), 'w') as f: json.dump(S, f, indent=2, default=str)
    return RES, W
print('core ready')

In [ ]:
import glob
TAGS = {'seasonal_naive': 'Seasonal Naive', 'persistence': 'Persistence (tuned)', 'lstm': 'LSTM', 'hybrid': 'Hybrid'}
def find_file(name):
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True) + glob.glob(f'./**/{name}', recursive=True)
    return hits[0] if hits else None
FR, HP = [], {}
for tag, label in TAGS.items():
    f = find_file(f'{tag}_predictions.csv')
    if f is None: print('MISSING predictions for', tag); continue
    d = pd.read_csv(f, parse_dates=['time']); d.insert(0, 'model', label); FR.append(d); print(f'{label:22s} {len(d):7d} rows  <- {f}')
    j = find_file(f'{tag}_best_hyperparameters.json')
    if j: HP[label] = json.load(open(j))
COMB = pd.concat(FR, ignore_index=True)
# every model must be evaluated on exactly the same Test hours
chk = COMB[(COMB.split == 'test')].groupby(['model', 'run', 'site', 'horizon']).time.agg(['count', 'min', 'max'])
print(chk.to_string())
ref = COMB[(COMB.split == 'test') & (COMB.run == 'run2')].groupby(['site', 'horizon']).time.nunique()
for m in COMB.model.unique():
    cnt = COMB[(COMB.split == 'test') & (COMB.run == 'run2') & (COMB.model == m)].groupby(['site', 'horizon']).time.nunique()
    assert (cnt == ref).all(), f'{m}: Test hours differ'
COMB.to_csv(os.path.join(OUT_DIR, 'rq2_combined_predictions.csv'), index=False)
ROWS = []
for m, g in COMB.groupby('model'):
    r = evaluate_all(g.drop(columns='model')); r.insert(0, 'model', m); ROWS.append(r)
RES = pd.concat(ROWS, ignore_index=True); RES.to_csv(os.path.join(OUT_DIR, 'rq2_results_all_models_long.csv'), index=False)
T = RES[(RES.group == 'Test') & RES.metric.isin(MAIN)]
W = T.pivot_table(index=['run', 'site', 'horizon', 'model'], columns='metric', values='value')[MAIN]
W.to_csv(os.path.join(OUT_DIR, 'rq2_results_all_models_test.csv'))
print('\n===== RQ2 TEST RESULTS (all models) =====\n', W.round(3).to_string())
SE = T[T.metric == 'MAE'].pivot_table(index=['run', 'site', 'horizon', 'model'], values='SE'); print('\nMAE bootstrap SE\n', SE.round(3).to_string())
# paired comparison: Hybrid vs each other model, MAE difference with block bootstrap (Test, run 2)
rng = np.random.default_rng(SEED); PR = []
for (site, h), g in COMB[(COMB.split == 'test') & (COMB.run == 'run2')].groupby(['site', 'horizon']):
    P = g.pivot_table(index='time', columns='model', values='pred'); A = g.groupby('time').actual.first(); E = (P.sub(A, axis=0)).abs()
    blocks = np.array_split(np.arange(len(E)), max(8, len(E) // BLOCK_H))
    for m in P.columns:
        if m == 'Hybrid': continue
        d = (E[m] - E['Hybrid']).values; bs = np.array([np.mean(np.concatenate([d[blocks[i]] for i in rng.integers(0, len(blocks), len(blocks))])) for _ in range(400)])
        PR.append(dict(site=site, horizon=h, vs=m, mae_diff=d.mean(), lo95=np.percentile(bs, 2.5), hi95=np.percentile(bs, 97.5)))
PR = pd.DataFrame(PR); PR.to_csv(os.path.join(OUT_DIR, 'rq2_paired_vs_hybrid.csv'), index=False); print('\nMAE(model) - MAE(Hybrid), run2 Test, 95% block-bootstrap CI\n', PR.round(3).to_string())
json.dump(HP, open(os.path.join(OUT_DIR, 'rq2_all_hyperparameters.json'), 'w'), indent=1, default=str)
print('\n===== HYPERPARAMETERS =====')
for k, v in HP.items(): print(k, json.dumps(v.get('best_hyperparameters', v), default=str)[:6000])
COL = {'Seasonal Naive': '#999999', 'Persistence (tuned)': '#E69F00', 'LSTM': '#0072B2', 'Hybrid': '#D55E00'}
fig, axs = plt.subplots(1, 2, figsize=(10, 3.6))
for ax, run in zip(axs, ['run1', 'run2']):
    d = W.xs(run, level='run')['MAE'].unstack('model'); d.index = [f'S{s} H={h}' for s, h in d.index]
    cols = [c for c in COL if c in d.columns]; d[cols].plot.bar(ax=ax, color=[COL[c] for c in cols], width=.8); ax.set(title=f'Test MAE, {run}', ylabel='kWh per hour'); ax.tick_params(axis='x', rotation=0, labelsize=8)
plt.tight_layout(); savefig('F1_rq2_test_mae')

## Paper figures (Fig. 4), drawn with seaborn/matplotlib from this notebook's own tables; PNG and CSV go to `OUT_DIR/paper_figures`.

In [ ]:
# ===== PAPER FIGURES: style chung (seaborn + matplotlib), giữ nhất quán giữa các notebook =====
import seaborn as sns
OKABE = {'blue': '#0072B2', 'orange': '#E69F00', 'green': '#009E73', 'red': '#D55E00', 'purple': '#CC79A7', 'sky': '#56B4E9', 'yellow': '#F0E442', 'grey': '#7F7F7F', 'black': '#222222'}
sns.set_theme(context='paper', style='ticks', font='DejaVu Sans', rc={
    'font.size': 7, 'axes.titlesize': 7.5, 'axes.labelsize': 7, 'xtick.labelsize': 6.5, 'ytick.labelsize': 6.5,
    'legend.fontsize': 6.5, 'legend.frameon': False, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.linewidth': 0.6, 'xtick.major.width': 0.6, 'ytick.major.width': 0.6, 'xtick.major.size': 2.5, 'ytick.major.size': 2.5,
    'lines.linewidth': 1.2, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left', 'figure.dpi': 110, 'savefig.dpi': 300})
FIG_DIR = os.path.join(OUT_DIR, 'paper_figures'); os.makedirs(FIG_DIR, exist_ok=True)
def panel_tag(ax, s):                                  # nhãn (a), (b), ... ở góc trên trái, như bài báo
    ax.text(-0.02, 1.08, s, transform=ax.transAxes, fontsize=8, fontweight='bold', va='bottom', ha='right')
def save_paper_fig(fig, name):
    p = os.path.join(FIG_DIR, name + '.png'); fig.savefig(p, bbox_inches='tight', pad_inches=0.03); print('Đã lưu', p)

In [ ]:
# ===== Fig. 4 của bài báo: tải trung bình theo giờ và MAE theo giờ (H = 24, Test, run 1), tính từ COMB của notebook này =====
SHORT = {'Seasonal Naive': 'Seasonal naive', 'Persistence (tuned)': 'Persistence', 'LSTM': 'LSTM', 'Hybrid': 'Hybrid'}
MCOL = {'Seasonal naive': OKABE['grey'], 'Persistence': OKABE['orange'], 'LSTM': OKABE['blue'], 'Hybrid': OKABE['red']}
Tt = COMB[(COMB.split == 'test') & (COMB.run == 'run1') & (COMB.horizon == 24)].copy()
Tt['hour'] = Tt.time.dt.hour; Tt['abs_err'] = (Tt.pred - Tt.actual).abs(); Tt['Model'] = Tt.model.map(SHORT)
LOAD = Tt[Tt.model == 'Hybrid'].groupby(['site', 'hour']).actual.mean().rename('mean_load').reset_index()
MAEH = Tt.groupby(['site', 'Model', 'hour']).abs_err.mean().rename('mae').reset_index()
LOAD.to_csv(os.path.join(OUT_DIR, 'fig4_hour_load.csv'), index=False); MAEH.to_csv(os.path.join(OUT_DIR, 'fig4_hour_mae_h24.csv'), index=False)
sites_ = sorted(Tt.site.unique())
fig, ax = plt.subplots(2, len(sites_), figsize=(4.8, 2.6), sharex=True, squeeze=False, gridspec_kw={'hspace': 0.32, 'wspace': 0.28})
for j, s in enumerate(sites_):
    sns.barplot(data=LOAD[LOAD.site == s], x='hour', y='mean_load', color='#BDBDBD', width=0.85, ax=ax[0, j])
    ax[0, j].set_title(f'Site {s}'); ax[0, j].set_ylabel('Mean load, kWh/h' if j == 0 else ''); ax[0, j].set_xlabel('')
    sns.lineplot(data=MAEH[MAEH.site == s], x='hour', y='mae', hue='Model', hue_order=list(MCOL), palette=MCOL, ax=ax[1, j], legend=False)
    ax[1, j].set_ylabel('Test MAE, kWh/h' if j == 0 else ''); ax[1, j].set_xlabel('Hour of day'); ax[1, j].set_ylim(bottom=0)
    ax[1, j].set_xticks([0, 6, 12, 18, 23]); ax[0, j].set_xticks([0, 6, 12, 18, 23]); ax[0, j].set_xticklabels([0, 6, 12, 18, 23])
fig.legend(handles=[plt.Line2D([], [], color=c, lw=1.4) for c in MCOL.values()], labels=list(MCOL), loc='lower center', ncol=4, bbox_to_anchor=(0.5, -0.13), fontsize=6)
panel_tag(ax[0, 0], '(a)'); panel_tag(ax[0, 1], '(b)'); panel_tag(ax[1, 0], '(c)'); panel_tag(ax[1, 1], '(d)')
save_paper_fig(fig, 'fig4_rq2_hour_profile'); plt.show()